# OpenPlaque — LAD Source-Space Plaque Transfer Validation v1

Fresh-baseline experiment. Runtime → Run all. The RCA reference-normalized plaque model is transferred to the LAD without LAD refitting.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT = DRIVE_ROOT / 'LAD_Source_Space_Plaque_Transfer_Validation_v1'
REUSE_VALID_CACHES = True
FORCE_RECOMPUTE = False
if FORCE_RECOMPUTE and OUTPUT.exists(): shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT/'notebook_started.json').write_text(json.dumps({'status':'started'}))
print('Drive root:', DRIVE_ROOT)
print('Output:', OUTPUT)
print('Reuse valid caches:', REUSE_VALID_CACHES)


In [ ]:
import os, shutil, sys
os.chdir('/content')
REPO = Path('/content/OpenPlaque_lad_plaque_transfer')
if REPO.exists(): shutil.rmtree(REPO)
BRANCH = 'lad-source-space-plaque-transfer-validation-from-main'
PINNED_SCIENCE_COMMIT = 'bf129196f645105897a954c90975278f95cec01d'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
print('Working directory repaired:', os.getcwd())
!git clone -q --branch $BRANCH https://github.com/pazzani/OpenPlaque.git $REPO
!git -C $REPO checkout -q $PINNED_SCIENCE_COMMIT
HEAD = get_ipython().getoutput(f'git -C {REPO} rev-parse HEAD')[0].strip()
MB = get_ipython().getoutput(f'git -C {REPO} merge-base HEAD {BASELINE}')[0].strip()
print('Checked out:', HEAD)
print('Merge base:', MB)
assert HEAD == PINNED_SCIENCE_COMMIT
assert MB == BASELINE
%pip install -q /content/OpenPlaque_lad_plaque_transfer
for k in list(sys.modules):
    if k == 'openplaque' or k.startswith('openplaque.'):
        del sys.modules[k]
os.chdir('/content')


In [ ]:
from openplaque.lad_source_space_plaque_transfer_validation_v1 import synthetic_self_test
print(synthetic_self_test())
!pytest -q /content/OpenPlaque_lad_plaque_transfer/tests/test_lad_source_space_plaque_transfer_validation_v1.py


In [ ]:
required = [
    DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
    DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
    DRIVE_ROOT/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
    DRIVE_ROOT/'LAD_Distal_Endpoint_Continuation_v1/best_independent_distal_extension.csv',
    DRIVE_ROOT/'LAD_Distal_Bidirectional_Validation_v1/summary.json',
    DRIVE_ROOT/'RCA_Source_Space_Plaque_Quantification_v1/RCA_source_space_station_quantification.csv',
    DRIVE_ROOT/'RCA_Source_Space_Plaque_Excess_Specificity_v1/summary.json',
    DRIVE_ROOT/'Longitudinal_Plaque_PCAT_Fusion_v1/LAD_source_longitudinal_plaque_profile_1mm.csv',
    DRIVE_ROOT/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing prerequisites:\n' + '\n'.join(missing))
(OUTPUT/'preflight_complete.json').write_text(json.dumps({
    'status':'complete','science_commit':PINNED_SCIENCE_COMMIT,'baseline':BASELINE,'required_count':len(required)
}, indent=2))
print('Preflight complete:', len(required), 'required artifacts found')


In [ ]:
from openplaque.lad_source_space_plaque_transfer_validation_v1 import run
result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
print(json.dumps(result['summary'], indent=2, default=str))
print('Report:', result['report'])
print('ZIP:', result['zip'])
